# Step 3: Data Cleaning
### Marketing Funnel Analysis: Bank Marketing Campaign Dataset

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/bank-additional-full.csv', sep=';')
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Dataset loaded: 41,188 rows × 21 columns


In [3]:
print("'Unknown' value counts per column:")
for col in df.columns:
    count = (df[col] == 'unknown').sum()
    if count > 0:
        pct = count / len(df) * 100
        print(f"  {col:<20} → {count:>5} unknowns ({pct:.1f}%)")

'Unknown' value counts per column:
  job                  →   330 unknowns (0.8%)
  marital              →    80 unknowns (0.2%)
  education            →  1731 unknowns (4.2%)
  default              →  8597 unknowns (20.9%)
  housing              →   990 unknowns (2.4%)
  loan                 →   990 unknowns (2.4%)


## Strategy for 'unknown' values
- `job` → keep, tag as 'unknown' (meaningful segment)
- `marital` → keep, tag as 'unknown' (only 0.2%)
- `education` → keep, tag as 'unknown' (meaningful segment)
- `default` → drop column (>20% unknown, not useful)
- `housing` → keep (unknowns are small)
- `loan` → keep (unknowns are small)
- `poutcome` → keep (unknown = no previous campaign, meaningful)

We will NOT drop rows, unknowns are a real customer segment.

In [4]:
df.drop(columns=['default'], inplace=True)
print("Dropped 'default' column")
print(f"Shape now: {df.shape}")

Dropped 'default' column
Shape now: (41188, 20)


In [5]:
df['y_binary'] = (df['y'] == 'yes').astype(int)
print("Target column encoded:")
print(df[['y', 'y_binary']].value_counts())

Target column encoded:
y    y_binary
no   0           36548
yes  1            4640
Name: count, dtype: int64


In [6]:
# These should be categorical
cat_cols = ['job', 'marital', 'education', 'housing', 
            'loan', 'contact', 'month', 'day_of_week', 
            'poutcome', 'y']

for col in cat_cols:
    df[col] = df[col].astype('category')

print("Categorical columns set:")
print(df.dtypes[df.dtypes == 'category'])

Categorical columns set:
job            category
marital        category
education      category
housing        category
loan           category
contact        category
month          category
day_of_week    category
poutcome       category
y              category
dtype: object


In [7]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[17, 25, 35, 45, 55, 65, 100],
    labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
)

print("Age group distribution:")
print(df['age_group'].value_counts().sort_index())

Age group distribution:
age_group
18-25     1661
26-35    14847
36-45    12844
46-55     8249
56-65     2963
65+        619
Name: count, dtype: int64


In [8]:
df['campaign_bucket'] = pd.cut(
    df['campaign'],
    bins=[0, 1, 3, 5, 100],
    labels=['1 contact', '2-3 contacts', '4-5 contacts', '6+ contacts']
)

print("Campaign intensity distribution:")
print(df['campaign_bucket'].value_counts().sort_index())

Campaign intensity distribution:
campaign_bucket
1 contact       17642
2-3 contacts    15911
4-5 contacts     4250
6+ contacts      3385
Name: count, dtype: int64


In [9]:
print("=== CLEANED DATASET SUMMARY ===")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nData types:\n{df.dtypes}")
print(f"\nAny nulls: {df.isnull().sum().sum()}")

=== CLEANED DATASET SUMMARY ===
Shape: 41,188 rows × 23 columns

Data types:
age                   int64
job                category
marital            category
education          category
housing            category
loan               category
contact            category
month              category
day_of_week        category
duration              int64
campaign              int64
pdays                 int64
previous              int64
poutcome           category
emp.var.rate        float64
cons.price.idx      float64
cons.conf.idx       float64
euribor3m           float64
nr.employed         float64
y                  category
y_binary              int64
age_group          category
campaign_bucket    category
dtype: object

Any nulls: 5


In [11]:
print("Null values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Null values per column:
age_group    5
dtype: int64


In [12]:
print("Ages that didn't fit into age_group bins:")
print(df[df['age_group'].isnull()]['age'].value_counts())

Ages that didn't fit into age_group bins:
age
17    5
Name: count, dtype: int64


In [13]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[16, 25, 35, 45, 55, 65, 100],
    labels=['17-25', '26-35', '36-45', '46-55', '56-65', '65+']
)

print("Fixed age group distribution:")
print(df['age_group'].value_counts().sort_index())

Fixed age group distribution:
age_group
17-25     1666
26-35    14847
36-45    12844
46-55     8249
56-65     2963
65+        619
Name: count, dtype: int64


In [15]:
print(f"Nulls remaining: {df.isnull().sum().sum()}")

Nulls remaining: 0


In [16]:
df.to_csv('../data/bank_cleaned.csv', index=False)
print("Cleaned dataset saved to data/bank_cleaned.csv")

Cleaned dataset saved to data/bank_cleaned.csv


## Cleaning Summary
- Dropped `default` column (too many unknowns, low value)
- Encoded `y` → `y_binary` (0/1) for calculations
- Set correct categorical data types
- Created `age_group` column for segmentation
- Created `campaign_bucket` column for funnel intensity analysis
- No rows dropped, all 41,188 records retained
- Dataset ready for funnel metric calculation